In [1]:
import pandas as pd

df = pd.read_csv("covid19_clinical_trials_cleaned.csv")

In [2]:
df['Conditions'].head(20)

0                                               Covid19
1                                  SARS-CoV-2 Infection
2                                               covid19
3                                                 COVID
4     Maternal Fetal Infection Transmission|COVID-19...
5                                              COVID 19
6                       Covid19|Exclusive Breastfeeding
7                                Covid19|Type2 Diabetes
8                                               Covid19
9     COVID-19 Infection|Cancer Patients|General Pop...
10                                             COVID 19
11                                              Covid19
12                                              Covid19
13                                              covid19
14                                              Covid19
15                    Hepatocellular Carcinoma|COVID-19
16                         Covid19|SARS-CoV-2 Infection
17                         IgG4 Related Disease|

In [3]:
# --- Conditions ---
conditions = (
    df[['NCT Number', 'Conditions']]
    .assign(Conditions=df['Conditions'].str.split('|'))
    .explode('Conditions')
)
conditions['Conditions'] = conditions['Conditions'].str.strip().str.lower()


conditions = conditions.rename(columns={'NCT Number': 'nct_number', 'Conditions': 'condition'})



In [4]:
covid_pattern = r'covid|sars[-\s]?cov|\bsars\b|severe acute respiratory syndrome|coronavirus|corona|ncov|cov-?2\b|cov-hi'
exclude_pattern = r'non-covid|non covid'

is_covid = (
    conditions['condition'].str.contains(covid_pattern, case=False, na=False, regex=True)
    & ~conditions['condition'].str.contains(exclude_pattern, case=False, na=False, regex=True)
)
conditions.loc[is_covid, 'condition'] = 'covid-19'

# ARDS after COVID, with word boundaries so "stewardship" is safe
is_ards = (
    conditions['condition'].str.contains(r'\bards\b|acute respiratory distress', case=False, na=False, regex=True)
    & (conditions['condition'] != 'covid-19')
)

conditions.loc[is_ards, 'condition'] = 'acute respiratory distress syndrome'


is_diabetes = conditions['condition'].str.match(
    r'^(diabetes|diabetes mellitus|diabete mellitus|diabete type 2|diabetes mellitus,? type [12]|'
    r'type ?[12] ?diabetes( mellitus)?)$', case=False, na=False)
conditions.loc[is_diabetes, 'condition'] = 'diabetes'
conditions = conditions.drop_duplicates(subset=['nct_number', 'condition'])

conditions = conditions.drop_duplicates(subset=['nct_number', 'condition'])
print(conditions['condition'].value_counts().head(10))

condition
covid-19                               4556
acute respiratory distress syndrome     211
anxiety                                 100
depression                               85
pneumonia, viral                         82
diabetes                                 70
pneumonia                                70
cancer                                   60
respiratory failure                      46
critical illness                         43
Name: count, dtype: int64


In [5]:
print(conditions['condition'].value_counts().head(20))


condition
covid-19                               4556
acute respiratory distress syndrome     211
anxiety                                 100
depression                               85
pneumonia, viral                         82
diabetes                                 70
pneumonia                                70
cancer                                   60
respiratory failure                      46
critical illness                         43
stress                                   40
quality of life                          37
healthy                                  34
cardiovascular diseases                  33
acute kidney injury                      25
obesity                                  25
cytokine storm                           25
infection viral                          24
stress, psychological                    24
pregnancy related                        23
Name: count, dtype: int64


In [6]:
conditions[conditions['condition'].str.contains(exclude_pattern, case=False, na=False, regex=True)]

,nct_number,condition
5140,NCT04550312,non-covid deaths


In [7]:
# --- Interventions (explode, then split Type: Detail) ---
interventions = (
    df[['NCT Number', 'Interventions']]
    .assign(Interventions=df['Interventions'].str.split('|'))
    .explode('Interventions')
)
interventions[['intervention_type', 'intervention_detail']] = (
    interventions['Interventions'].str.split(':', n=1, expand=True)
)
interventions['intervention_type'] = interventions['intervention_type'].str.strip()
interventions['intervention_detail'] = interventions['intervention_detail'].str.strip()
interventions = interventions.rename(columns={'NCT Number': 'nct_number'}).drop(columns=['Interventions'])

print(interventions.head(10))


    nct_number intervention_type  \
0  NCT04785898   Diagnostic Test   
1  NCT04595136              Drug   
1  NCT04595136              Drug   
2  NCT04395482             Other   
3  NCT04416061   Diagnostic Test   
4  NCT04395924   Diagnostic Test   
5  NCT04516954        Biological   
6  NCT04476940        Behavioral   
7  NCT04634214               NaN   
8  NCT04602884   Diagnostic Test   

                                 intervention_detail  
0                    ID Now™ COVID-19 Screening Test  
1                              Drug COVID19-0001-USR  
1                                      normal saline  
2         Lung CT scan analysis in COVID-19 patients  
3                           COVID 19 Diagnostic Test  
4  Diagnosis of SARS-Cov2 by RT-PCR and : IgG, Ig...  
5                       Convalescent COVID 19 Plasma  
6                     COVID-19 Breastfeeding Support  
7                                                NaN  
8  Breath biopsy sampling using the ReCIVA Breath... 

In [8]:


# --- Funded Bys ---
funders = (
    df[['NCT Number', 'Funded Bys']]
    .assign(**{'Funded Bys': df['Funded Bys'].str.split('|')})
    .explode('Funded Bys')
)
funders = funders.rename(columns={'NCT Number': 'nct_number', 'Funded Bys': 'funder'})

print(funders.head(11))


     nct_number    funder
0   NCT04785898     Other
1   NCT04595136     Other
2   NCT04395482     Other
3   NCT04416061  Industry
4   NCT04395924     Other
5   NCT04516954     Other
6   NCT04476940     Other
7   NCT04634214     Other
8   NCT04602884  Industry
9   NCT04384588     Other
10  NCT04355897     Other


In [9]:

# --- Phases ---
phases = (
    df[['NCT Number', 'Phases']]
    .assign(Phases=df['Phases'].str.split('|'))
    .explode('Phases')
)
phases['Phases'] = phases['Phases'].str.strip()
phases = phases.rename(columns={'NCT Number': 'nct_number', 'Phases': 'phase'})

print(phases.head(10))


    nct_number           phase
0  NCT04785898  Not Applicable
1  NCT04595136         Phase 1
1  NCT04595136         Phase 2
2  NCT04395482  Not Applicable
3  NCT04416061  Not Applicable
4  NCT04395924  Not Applicable
5  NCT04516954   Early Phase 1
6  NCT04476940  Not Applicable
7  NCT04634214  Not Applicable
8  NCT04602884  Not Applicable


In [10]:
before = len(conditions)
conditions = conditions.drop_duplicates(subset=['nct_number', 'condition'])
print(before, '->', len(conditions))
print(conditions['condition'].value_counts().head(5))

10069 -> 10069
condition
covid-19                               4556
acute respiratory distress syndrome     211
anxiety                                 100
depression                               85
pneumonia, viral                         82
Name: count, dtype: int64


In [11]:
interventions['intervention_type'] = interventions['intervention_type'].fillna('No intervention (observational)')

In [12]:

# --- Save all tables ---
df.to_csv('covid19_clinical_trials_normalized.csv', index=False)
conditions.to_csv('conditions.csv', index=False)
interventions.to_csv('interventions.csv', index=False)
funders.to_csv('funders.csv', index=False)
phases.to_csv('phases.csv', index=False)

for name, table in [('conditions', conditions), ('interventions', interventions), 
                     ('funders', funders), ('phases', phases)]:
    print(f"--- {name} ---")
    print(table.head(3))
    print(table.shape)
    print()

--- conditions ---
    nct_number condition
0  NCT04785898  covid-19
1  NCT04595136  covid-19
2  NCT04395482  covid-19
(10069, 2)

--- interventions ---
    nct_number intervention_type              intervention_detail
0  NCT04785898   Diagnostic Test  ID Now™ COVID-19 Screening Test
1  NCT04595136              Drug            Drug COVID19-0001-USR
1  NCT04595136              Drug                    normal saline
(9051, 3)

--- funders ---
    nct_number funder
0  NCT04785898  Other
1  NCT04595136  Other
2  NCT04395482  Other
(6385, 2)

--- phases ---
    nct_number           phase
0  NCT04785898  Not Applicable
1  NCT04595136         Phase 1
1  NCT04595136         Phase 2
(6175, 2)

